# Setup and Initialisation

In [ ]:
import csv
import os

def initialise_file():
    """Safely builds the foundation files with exact headers if they don't exist yet."""
    if not os.path.exists("users.csv") or os.path.getsize("users.csv") == 0:
        with open("users.csv", "w", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)
            writer.writerow(["id", "name", "email", "password", "role", "skills"])
            
    if not os.path.exists("jobs.csv") or os.path.getsize("jobs.csv") == 0:
        with open("jobs.csv", "w", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)
            writer.writerow(["id", "employer_id", "title", "description", "skills", "experience"])
            
    if not os.path.exists("applications.csv") or os.path.getsize("applications.csv") == 0:
        with open("applications.csv", "w", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)
            writer.writerow(["id", "job_id", "employee_id"])

# User Management Functions

In [ ]:
def save_user(name,email,password,role,skills):

    try:

        users = []

        with open("users.csv","r",newline="",encoding="utf-8") as file:
            reader = csv.DictReader(file)
            users = list(reader)

        for user in users:

            if user["email"] == email:
                return False

        with open("users.csv","a",newline="",encoding="utf-8") as file:

            writer = csv.DictWriter(
                file,
                fieldnames=[
                    "id","name","email",
                    "password","role","skills"
                ]
            )

            writer.writerow({
                "id":len(users)+1,
                "name":name,
                "email":email,
                "password":password,
                "role":role,
                "skills":skills
            })

        return True

    except PermissionError:
        return "LOCKED"
def validate_login(email, password):
    """Checks credentials and returns the user dictionary if successful."""
    if not os.path.exists("users.csv"): 
        return None
        
    with open("users.csv", "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
            if row["email"] == email and row["password"] == password:
                return {
                    "id": row["id"],
                    "name": row["name"],
                    "role": row["role"],
                    "skills": row["skills"]
                }
    return None

# Job Management Functions

In [ ]:
def save_job(employer_id, title, description, skills, experience):
    try:
        job_id = 1
        if os.path.exists("jobs.csv"):
            with open("jobs.csv", "r", newline="", encoding="utf-8") as file:
                rows = list(csv.reader(file))
                if len(rows) > 1:
                    try:
                        job_id = int(rows[-1][0]) + 1
                    except ValueError:
                        job_id = 1

        with open("jobs.csv", "a", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)
            writer.writerow([job_id, employer_id, title, description, skills, experience])
        return True
    except PermissionError:
        return "LOCKED"

def get_employer_jobs(employer_id):
    """Fetches only the jobs posted by a specific employer ID."""
    jobs = []
    if not os.path.exists("jobs.csv"): 
        return jobs
        
    with open("jobs.csv", "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
            if str(row["employer_id"]) == str(employer_id):
                jobs.append(row)
    return jobs

def get_jobs():
    """Fetches all available job listings for the Job Seeker explorer panel."""
    jobs = []
    if not os.path.exists("jobs.csv"): 
        return jobs
        
    with open("jobs.csv", "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
            jobs.append(row)
    return jobs

# Job Application Functions

In [ ]:
def save_application(job_id, employee_id):

    applications = []

    if os.path.exists("applications.csv"):

        with open("applications.csv","r",newline="",encoding="utf-8") as file:

            reader = csv.DictReader(file)

            applications = list(reader)

    for app in applications:

        if (
            str(app["job_id"]) == str(job_id)
            and
            str(app["employee_id"]) == str(employee_id)
        ):
            return False

    app_id = len(applications)+1

    with open("applications.csv","a",newline="",encoding="utf-8") as file:

        writer = csv.writer(file)

        writer.writerow([
            app_id,
            job_id,
            employee_id
        ])

    return True

def get_applications(employee_id):

    if not os.path.exists("applications.csv"):
        return []

    with open(
        "applications.csv",
        "r",
        newline="",
        encoding="utf-8"
    ) as file:

        reader = csv.DictReader(file)

        return [
            app for app in reader
            if str(app["employee_id"]) == str(employee_id)
        ]

def get_job_applicants(job_id):
    """Maps applications back to user profiles so Employers can review candidates."""
    applicant_ids = []
    if os.path.exists("applications.csv"):
        with open("applications.csv", "r", newline="", encoding="utf-8") as file:
            reader = csv.DictReader(file)
            for row in reader:
                if str(row["job_id"]) == str(job_id):
                    applicant_ids.append(str(row["employee_id"]))

    applicants = []
    if os.path.exists("users.csv") and applicant_ids:
        with open("users.csv", "r", newline="", encoding="utf-8") as file:
            reader = csv.DictReader(file)
            for row in reader:
                if str(row["id"]) in applicant_ids:
                    applicants.append(row)
    
    return applicants